# AMADS Coding Notebooks

## Chord Loops

What chord progressions are common?
How can we measure similarity among progressions?

---

**By (author/s):** Mark Gotham

**For:** Attached to the
[AMADS code library](https://github.com/music-computing/amads/) and
["Keeping Score" book](https://doi.org/10.5281/zenodo.14938027),
but open to all.

**Licence:** MIT.

**Colour key:**
- <font color='green'> Green is for a block of information.
- <font color='purple'> Purple is for an exercise.
- <font color='crimson'> Crimson is for the solution to the exercise above it.

## <font color='green'> Chord Loops in the "McGill Billboard" dataset

The McGill-Billboard dataset is a human-made annotation of the chords, structure (and more) in popular songs.

"Popularity" is given by inclusion in the Billboard chart (measured by them according to sales, streams).

In this notebook, we parse these chord annotations,
extract repeating progressions (aka loops),
and explore various measures of how similar two loops really are.


### Import packages

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
import re
import seaborn as sns

### GET THE DATA (and point to it ...)

- Use you download the data used here from [this direct link](https://www.dropbox.com/scl/fi/0iiwc4fq0d469sjn5glw4/billboard-2.0-salami_chords.tar.xz?rlkey=7x2ptmu3zr01sf2nl75mu27by&dl=1).
- Store it next to this notebook or in some other known location which you can access from here.

In [ ]:
you_are_here = Path(".")
you_are_here.resolve() # Check this is what you expect E.g., "Users/<name>/Documents" ...

The following cell assumes you have the dataset in a folder called 'McGill-Billboard' that is located alongside this notebook

If that's not the case, adjust as needed for your path:


In [ ]:
[x for x in you_are_here.iterdir() if x.is_dir()]

In [ ]:
DATA_DIR = you_are_here / 'McGill-Billboard'
DATA_DIR.resolve()

If using colab, make sure the data is stored beside this notebook, uncomment and run the next cells:

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# # Then there's an authentication page after which you should be able to access
# you_are_here = Path('/content/drive/My Drive/')
# list(you_are_here.iterdir())

In [ ]:
# # If you see 'McGill-Billboard' listed, then go ahead with
# DATA_DIR = you_are_here / 'McGill-Billboard'

---

## <font color='purple'> Exercise 1. Dataset Metadata

Before we get into the chord data, let's explore the metadata.

Read all file headers and summarise dataset-level metadata.

Focus on time signature distribution; plot a comparison for how often each is used.

In [ ]:
def read_metadata(path: Path) -> dict:
    """Extract # header lines from a single SALAMI file."""
    pass
    # your code here ;)

## <font color='crimson'> Solution 1. Dataset Metadata

In [ ]:
def read_metadata(path: Path) -> dict:
    """Extract # header lines from a single SALAMI file."""
    meta = {'file': path.stem}
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line.startswith('#'):
            break   # headers are always at the top
        m = re.match(r'#\s*(\w+):\s*(.+)', line)
        if m:
            meta[m.group(1).lower()] = m.group(2).strip()
    return meta

records = [read_metadata(p) for p in sorted(DATA_DIR.rglob('*.txt'))]
meta_df = pd.DataFrame(records)

print(f"Files loaded : {len(meta_df)}")
print(f"Columns      : {list(meta_df.columns)}")
print()
print(meta_df.head())

# Time signature bar charts:
if 'metre' in meta_df.columns:
    metre_counts = meta_df['metre'].fillna('unknown').value_counts()
else:
    metre_counts = pd.Series({'unknown': len(meta_df)})

minority_counts = metre_counts[metre_counts.index != '4/4']

def annotate_bars(ax):
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.3,
                str(int(bar.get_height())),
                ha='center', va='bottom', fontsize=10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Left: all signatures
metre_counts.plot(kind='bar', ax=ax1, color='steelblue', edgecolor='white')
ax1.set_title('All time signatures')
ax1.set_xlabel('Time signature')
ax1.set_ylabel('Number of songs')
ax1.tick_params(axis='x', rotation=0)
annotate_bars(ax1)

# Right: excluding 4/4
minority_counts.plot(kind='bar', ax=ax2, color='coral', edgecolor='white')
ax2.set_title('Excluding 4/4')
ax2.set_xlabel('Time signature')
ax2.set_ylabel('Number of songs')
ax2.tick_params(axis='x', rotation=0)
annotate_bars(ax2)

fig.suptitle('Time signature distribution', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n4/4 accounts for {metre_counts.get('4/4', 0) / metre_counts.sum():.1%} of the dataset.")

Apart from the obvious fact that 4/4 dominates, 12/8 is surprisingly common.

---
## <font color='purple'> Exercise 2. Loading, Parsing & Normalising Chords

- Parse the SALAMI chord files into a structured DataFrame.
- Write a parse_file(path) function that reads a single `.txt` file, extracts metadata from #-prefixed header lines, and collects chord lists from the body.
- Write `load_dataset(data_dir)` to apply it across all files.
- Finally, write a `normalise_chord`routine to convert SALAMI chord strings (e.g. 'Bb:maj') into Chord objects (dropping non-chord tokens like 'N' and 'X'), and `collapse_repeats` to remove consecutive duplicate chords from a progression.

In [ ]:
def parse_file(path: Path) -> dict:
    """Parse a single SALAMI chord file into structured data."""
    pass

In [ ]:
def load_dataset(data_dir: Path) -> pd.DataFrame:
    pass

In [ ]:
from amads.core.chord import Chord

def normalise_chord(chord_str: str) -> Chord | None:
    pass


def normalise_progression(chords: list[str]) -> list[tuple[str,str]]:
    """Normalise a list of SALAMI chord strings; drop unparseable entries."""
    pass


def collapse_repeats(prog: list) -> list:
    """Collapse consecutive identical chords:  [C,C,G,G] → [C,G]."""
    pass

## <font color='crimson'> Solution 2. Loading, Parsing & Normalising Chords

In [ ]:
SECTION_RE = re.compile(r'([A-Z](?:,\s*[\w-]+)*),\s*')
CHORD_RE   = re.compile(r'\|([^|]+)\|')

def parse_file(path: Path) -> dict:
    """Parse a single SALAMI chord file into structured data."""
    meta = {}
    rows = []
    text = path.read_text(encoding='utf-8')

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        # Metadata headers
        if line.startswith('#'):
            m = re.match(r'#\s*(\w+):\s*(.+)', line)
            if m:
                meta[m.group(1)] = m.group(2).strip()
            continue
        # Skip silence / end markers
        if re.search(r'\b(silence|end)\b', line, re.I):
            continue

        parts = line.split('\t', 1)
        if len(parts) < 2:
            continue
        timestamp = float(parts[0])
        rest = parts[1]

        # Strip trailing annotations in parens
        rest = re.sub(r',?\s*\(.*?\)', '', rest)
        rest = re.sub(r',?\s*voice.*$', '', rest, flags=re.I)
        rest = re.sub(r',?\s*fadeout.*$', '', rest, flags=re.I)

        # Extract section label if present
        section_label = None
        sm = re.match(r'^([A-Z])(?:,\s*([\w\-]+))?,\s*', rest)
        if sm:
            section_label = sm.group(2) or sm.group(1)
            rest = rest[sm.end():]

        # Extract chord bars
        chords = [c.strip() for c in CHORD_RE.findall(rest)]
        if not chords:
            continue

        rows.append({
            'timestamp': timestamp,
            'section':   section_label,
            'chords':    chords,
        })

    return {'meta': meta, 'rows': rows}


def load_dataset(data_dir: Path) -> pd.DataFrame:
    records = []
    for path in sorted(data_dir.rglob('*.txt')):
        parsed = parse_file(path)
        meta   = parsed['meta']
        for row in parsed['rows']:
            records.append({
                'file':    path.stem,
                'title':   meta.get('title', ''),
                'artist':  meta.get('artist', ''),
                'tonic':   meta.get('tonic', ''),
                'metre':   meta.get('metre', ''),
                'section': row['section'],
                'chords':  row['chords'],
            })
    return pd.DataFrame(records)


# Demo with the single sample file
sample = parse_file(DATA_DIR / "0003" / 'salami_chords.txt')
df_sample = pd.DataFrame(sample['rows'])
print(sample['meta'])
df_sample.head(10)

In [ ]:
from amads.core.chord import Chord
def normalise_chord(chord_str: str) -> Chord | None:
    chord_str = chord_str.strip()
    if chord_str in ('N', 'X', '', '*'):
        return None
    if ':' in chord_str:
        root, qual = chord_str.split(':', 1)
    else:
        root, qual = chord_str, 'maj'
    qual = re.sub(r'\(.*?\)', '', qual.split('/')[0])
    try:
        return Chord(root, qual)
    except ValueError:
        return None

def normalise_progression(chords: list[str]) -> list[tuple[str,str]]:
    """Normalise a list of SALAMI chord strings; drop unparseable entries."""
    result = [normalise_chord(c) for c in chords]
    return [c for c in result if c is not None]


def collapse_repeats(prog: list) -> list:
    """Collapse consecutive identical chords:  [C,C,G,G] → [C,G]."""
    if not prog:
        return prog
    out = [prog[0]]
    for chord in prog[1:]:
        if chord != out[-1]:
            out.append(chord)
    return out


# Quick test
test_chords = ['Bb:maj', 'Bb:maj', 'Eb:maj', 'F:7']
normed = normalise_progression(test_chords)
print('Normalised:', normed)
print('Collapsed: ', collapse_repeats(normed))

---

## <font color='purple'> Exercise 3. Extracting Loops

Each file in this dataset has one row per blocks.
That is, the dataset is pre-segmented into small blocks (c.4 to a verse).
These may correspond to phrases or sub-sections.
They often correspond to a chord loop
(e.g., the 4 chords in one row are repeated in the next).

Let's look at the chords in these short sections and explore them as possible loops.

For the one timestamped section row,
we'll normalise and repeat-collapse into a tuple (hashable, usable as a dict key).

## <font color='crimson'> Solution 3. Extracting Loops

In [ ]:
def extract_loops(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add columns  `norm_chords`  (list of (root,qual) tuples)
    and  `loop`  (collapsed, hashable tuple) to the dataframe.
    """
    df = df.copy()
    df['norm_chords'] = df['chords'].apply(normalise_progression)
    df['loop']        = df['norm_chords'].apply(
        lambda p: tuple(collapse_repeats(p))
    )
    df = df[df['loop'].apply(len) > 0]   # drop empty
    return df


df_sample = extract_loops(df_sample)
df_sample[['section','loop']].head(12)

---

## <font color='purple'> Exercise 4. Catalogue Exploration

Load the dataset to explore.
In the cell below, chose between:
- `df_sample` (defined above, good for initial testing)
- full dataset (uncomment and run when ready for all)

In [ ]:
# df = df_sample  # <<< A sample for testing

df = extract_loops(load_dataset(DATA_DIR))  # <<< Full, when ready

## <font color='crimson'> Solution 4. Catalogue Exploration

In [ ]:
# Most common loops
# Most common loops
loop_counts = Counter(df['loop'])
top_loops = pd.DataFrame(loop_counts.most_common(20), columns=['loop', 'count'])
top_loops['label'] = top_loops['loop'].apply(
    lambda prog: ' → '.join(c.label for c in prog)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Top loops bar chart
axes[0].barh(top_loops['label'][:10][::-1], top_loops['count'][:10][::-1])
axes[0].set_title('Top 10 loops (by frequency)')
axes[0].set_xlabel('Count')

# Section type distribution
section_counts = df['section'].value_counts().head(12)
axes[1].bar(section_counts.index, section_counts.values)
axes[1].set_title('Section type distribution')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\nTotal rows: {len(df)}")
print(f"Unique loops: {df['loop'].nunique()}")

---
## <font color='purple'> Exercise 5. Equivalence & Similarity

### 5a. Chord lookup table → pitch-class vectors

Represent chords as a **12-dimensional binary vectors** over pitch classes (C-B, 0-11).

Measuring 'similarity' in music is always complicated.

One reasonable approximation for chords is the dot product between these vectors, normalised by set sizes.

## <font color='crimson'> Solution 5a

In [ ]:
# AMADS has a Pitch class with an inbuild pitch_class attribute
from amads.core.pitch import Pitch
p = Pitch("E4")
p

In [ ]:
# Pitch-class lookup ('C' -> 0,'C#' -> 1, ...)
p.pitch_class

In [ ]:
# And here's the chord vector

from amads.core.vector_transforms_checks import indices_to_indicator, rotate

# Intervals from root for each quality  (semitones)
QUALITY_INTERVALS = {
    'maj':   [0, 4, 7],
    'min':   [0, 3, 7],
    'dim':   [0, 3, 6],
    'aug':   [0, 4, 8],
    '7':     [0, 4, 7, 10],
    'maj7':  [0, 4, 7, 11],
    'min7':  [0, 3, 7, 10],
    'hdim7': [0, 3, 6, 10],
    'dim7':  [0, 3, 6, 9],
}

indices_to_indicator(QUALITY_INTERVALS["maj"], indicator_length=12)

In [ ]:
from amads.core.chord import QUALITIES
def chord_vector(chord: Chord, strip_seventh: bool = False) -> np.ndarray:
    if strip_seventh and len(chord) == 4:
        intervals = QUALITIES[chord.quality][:3]
        root_pc = chord.root.pitch_class
        pcs = [(root_pc + i) % 12 for i in intervals]
        vec = np.zeros(12, dtype=float)
        for pc in pcs:
            vec[pc] = 1.0
        return vec
    return np.array(chord.pitch_class_vector, dtype=float)

def chord_similarity(c1: Chord, c2: Chord, strip_seventh: bool = False) -> float:
    """
    Shared-note similarity between two (root, quality) chords.
    Returns  |intersection| / |union|  (Jaccard on pitch classes).
    """
    v1 = chord_vector(c1, strip_seventh=strip_seventh)
    v2 = chord_vector(c2, strip_seventh=strip_seventh)
    intersection = np.dot(v1, v2)
    union = np.sum(np.clip(v1 + v2, 0, 1))
    return float(intersection / union) if union > 0 else 0.0

# Example from our discussion
cmaj = Chord('C', 'maj')
amin = Chord('A', 'min')
print(f"C major notes: {np.where(chord_vector(cmaj))}")
print(f"A minor notes: {np.where(chord_vector(amin))}")
print(f"Chord similarity C:maj ↔ A:min = {chord_similarity(cmaj, amin):.3f}")
print(f"Chord similarity C:maj ↔ C:7   = {chord_similarity(Chord('C','maj'), Chord('C','7')):.3f}")
print(f"  (strip_seventh=True)          = {chord_similarity(Chord('C','maj'), Chord('C','7'), strip_seventh=True):.3f}")

## <font color='purple'> Exercise 5b. Progression-level similarity

Using your answer to 5a, establish identity, equivalence, and similarity among progressions.

In [ ]:
%pip install scikit-learn

## <font color='crimson'> Solution 5b. Progression-level similarity

In [ ]:
def progression_vector(prog: tuple, strip_seventh: bool = False) -> np.ndarray:
    """
    Stack chord vectors into a (len, 12) matrix.
    Useful for rotation-aware comparison.
    """
    return np.array([chord_vector(c, strip_seventh=strip_seventh) for c in prog])

def _align_score(a: np.ndarray, b: np.ndarray) -> float:
    """
    Mean Jaccard similarity for two same-length vector matrices
    (intersection / union).

    Parameters
    ----------
    a : np.ndarray
    b : np.ndarray

    Return
    ------
    float

    Examples
    --------
    >>> a = np.array([0, 1, 2]) ...
    """
    sims = []
    for i in range(len(a)):
        inter = np.dot(a[i], b[i])
        union = np.sum(np.clip(a[i]+b[i], 0, 1))
        sims.append(float(inter/union) if union > 0 else 0.0)
    return float(np.mean(sims))


def progression_similarity(
        prog_a: tuple,
        prog_b: tuple,
        strip_seventh: bool = False,
        rotation_invariant: bool = True
) -> float:
    """
    Mean shared-note similarity across paired chords.

    If lengths differ, the shorter is zero-padded.
    If `rotation_invariant=True`, tries all rotations of the shorter
    progression and returns the best alignment score.
    """
    # Pad shorter progression with zero vectors
    n = max(len(prog_a), len(prog_b))
    def pad(prog):
        mat = progression_vector(prog, strip_seventh)
        pad_rows = n - len(prog)
        return np.vstack([mat, np.zeros((pad_rows, 12))]) if pad_rows else mat

    va, vb = pad(prog_a), pad(prog_b)

    if not rotation_invariant:
        return _align_score(va, vb)

    # Try all rotations of vb
    best = max(_align_score(va, np.roll(vb, k, axis=0))
               for k in range(n))
    return best

# Demo
p1 = (Chord('C','maj'), Chord('G','maj'), Chord('A','min'), Chord('F','maj'))
p2 = (Chord('G','maj'), Chord('A','min'), Chord('F','maj'), Chord('C','maj'))
p3 = (Chord('C','maj'), Chord('G','maj'), Chord('A','min'), Chord('E','maj'))

print(f"p1 vs p2 (rotation ok):  {progression_similarity(p1, p2, rotation_invariant=True):.3f}  (expect c.1.0)")
print(f"p1 vs p2 (no rotation):  {progression_similarity(p1, p2, rotation_invariant=False):.3f}")
print(f"p1 vs p3 (swap):         {progression_similarity(p1, p3):.3f}")

## <font color='purple'> Exercise 5c. Boolean matchers

Implement boolean equivalence checks for chord progressions:
- **is_rotation_equivalent**: is `b` a cyclic rotation of `a`?
- **chord_match**: do two chords share root and triadic quality (with optional seventh tolerance)?
- **transpose_equivalent**: is `b` a transposition of `a`, and by how many semitones?
- **neighbour_swap_distance**: minimum adjacent swaps to reorder `a` into `b`.

The cell below suggests AMADS functionality for part of this solution.

In [ ]:
from amads.core.vector_transforms_checks import is_rotation_equivalent
from amads.core.edit import neighbour_swap_distance

In [ ]:
def chord_match(c1: Chord, c2: Chord, allow_seventh: bool = True) -> bool:
    """Your code here"""
    pass


## <font color='crimson'> Solution 5c. Boolean matchers

In [ ]:
def chord_match(c1: Chord, c2: Chord, allow_seventh: bool = True) -> bool:
    if c1 == c2:
        return True
    if c1.root is None or c2.root is None:
        return False
    if c1.root.pitch_class != c2.root.pitch_class:
        return False
    if allow_seventh:
        # compare base triad quality only
        TRIAD_BASE = {
            'dominant7': 'major', 'major7': 'major',
            'minor7': 'minor', 'diminished7': 'diminished',
            'half-dim7': 'diminished',
        }
        q1 = TRIAD_BASE.get(c1.quality, c1.quality)
        q2 = TRIAD_BASE.get(c2.quality, c2.quality)
        return q1 == q2
    return c1.quality == c2.quality

def transpose_equivalent(a, b, allow_seventh=True):
    if len(a) != len(b):
        return False, 0
    shifts = {(cb.root.pitch_class - ca.root.pitch_class) % 12
              for ca, cb in zip(a, b)}
    if len(shifts) != 1:
        return False, 0
    shift = shifts.pop()
    qualities_match = all(
        chord_match(Chord('C', ca.quality), Chord('C', cb.quality),
                    allow_seventh=allow_seventh)
        for ca, cb in zip(a, b)
    )
    return qualities_match, shift

In [ ]:
p1        = tuple(Chord(*c) for c in (('C','maj'), ('G','maj'), ('D','min'), ('A','min')))
p2        = tuple(Chord(*c) for c in (('C','maj'), ('D','min'), ('G','maj'), ('A','min')))  # One swap
rotated_p1 = tuple(Chord(*c) for c in (('G','maj'), ('D','min'), ('A','min'), ('C','maj')))

print("is_rotation_equivalent(p1, rotated p1):",
      is_rotation_equivalent(p1, rotated_p1))

print("chord_match C:maj ↔ C:7  (allow_seventh=True): ",
      chord_match(Chord('C','maj'), Chord('C','7')))
print("chord_match C:maj ↔ C:7  (allow_seventh=False):",
      chord_match(Chord('C','maj'), Chord('C','7'), allow_seventh=False))

eq, shift = transpose_equivalent(
    p1,
    tuple(Chord(*c) for c in (('D','maj'), ('A','maj'), ('E','min'), ('B','min')))
)
print(f"transpose_equivalent (up a tone): {eq}, shift={shift} semitones")

In [ ]:
print(f"neighbour_swap_distance(p1, p2) = {neighbour_swap_distance(p1, p2)}")

## <font color='purple'> 6. Pattern Mining:

### <font color='purple'> Exercise 6a. Chord transition bigrams

Retrieve pairs of chords within rows.
Rows with only one chord are very common, so filter for cases with 2+ different chords.

### <font color='crimson'> Solution 6a. Chord transition bigrams

For this we'll use `pairwise` from `itertools`.

This requires Python 3.10+. If you're on an older version; use `zip(x,x[1:])`.

In [ ]:
from itertools import pairwise

In [ ]:
def get_bigrams(df: pd.DataFrame) -> Counter:
    bigrams = Counter()
    for loop in df['loop']:
        # treat as cyclic: last chord → first chord
        cyclic = loop + (loop[0],)
        bigrams.update(pairwise(cyclic))
    return bigrams

bigrams = get_bigrams(df)
print("Top 10 transitions:")
for chord_pair, count in bigrams.most_common(10):
    print(f"{chord_pair[0].label} → {chord_pair[1].label}  ({count})")

## <font color='purple'> 6b. Transition matrix heatmap

Build a chord-to-chord transition count matrix for the top_n chords.

In [ ]:
def transition_matrix(
        df: pd.DataFrame,
        top_n: int = 12
) -> pd.DataFrame:
    """Build a chord-to-chord transition count matrix for the top_n chords."""
    pass
    # Your code here ;)

## <font color='crimson'> 6b. Transition matrix heatmap

In [ ]:
def transition_matrix(df: pd.DataFrame,
                      top_n: int = 12) -> pd.DataFrame:
    """Build a chord-to-chord transition count matrix for the top_n chords."""
    bigrams = get_bigrams(df)
    # Most frequent individual chords
    chord_counts = Counter(c for loop in df['loop'] for c in loop)
    top_chords   = [c for c, _ in chord_counts.most_common(top_n)]
    labels = [c.label for c in top_chords]

    mat = np.zeros((top_n, top_n))
    for i, ca in enumerate(top_chords):
        for j, cb in enumerate(top_chords):
            mat[i, j] = bigrams.get((ca, cb), 0)
    # Row-normalise to probabilities
    row_sums = mat.sum(axis=1, keepdims=True)
    mat = np.divide(mat, row_sums, where=row_sums > 0)

    return pd.DataFrame(mat, index=labels, columns=labels)


tmat = transition_matrix(df, top_n=6)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(tmat, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax)
ax.set_title('Chord transition probabilities (row → column)')
ax.set_xlabel('To')
ax.set_ylabel('From')
plt.tight_layout()
plt.show()

## <font color='purple'> 6c. Similarity matrix across loops

Compute pairwise progression_similarity for a list of loops.

In [ ]:
def loop_similarity_matrix(
        loops: list[tuple],
        strip_seventh: bool = False,
        rotation_invariant: bool = True
) -> np.ndarray:
    """Compute pairwise progression_similarity for a list of loops."""
    pass # Your code here ;)

## <font color='crimson'> 6c. Similarity matrix across loops

In [ ]:
def loop_similarity_matrix(
        loops: list[tuple],
        strip_seventh: bool = False,
        rotation_invariant: bool = True
) -> np.ndarray:
    """Compute pairwise progression_similarity for a list of loops."""
    n = len(loops)
    mat = np.eye(n)
    for i in range(n):
        for j in range(i+1, n):
            s = progression_similarity(loops[i], loops[j],
                                       strip_seventh=strip_seventh,
                                       rotation_invariant=rotation_invariant)
            mat[i, j] = mat[j, i] = s
    return mat

# Use top unique loops from the dataset, excluding rows with one chord only (not really a "loop")
top_unique = [
    loop for loop, _ in Counter(df['loop']).most_common(50)
    if len(loop) >= 2
][:15]

sim_mat    = loop_similarity_matrix(top_unique, rotation_invariant=True)
labels = ['→'.join(c.label for c in p) for p in top_unique]

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(sim_mat, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=labels, yticklabels=labels,
            vmin=0, vmax=1, ax=ax)
ax.set_title('Pairwise loop similarity (shared notes, rotation-invariant)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=45, fontsize=8)
plt.tight_layout()
plt.show()

## <font color='purple'> 6d. Clustering by similarity
m
Prepare and plot a dendogram for clusters in this data.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

## <font color='crimson'> 6d. Clustering by similarity

In [ ]:
dist_mat = 1.0 - sim_mat
np.fill_diagonal(dist_mat, 0)
condensed = squareform(dist_mat)

Z = linkage(condensed, method='average')

fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(Z, labels=labels, ax=ax, leaf_rotation=45, leaf_font_size=8)
ax.set_title('Hierarchical clustering of loops by shared-note similarity')
plt.tight_layout()
plt.show()

# Flat clusters at a chosen distance threshold
threshold = 0.4
cluster_ids = fcluster(Z, t=threshold, criterion='distance')
for loop, cid in sorted(zip(top_unique, cluster_ids), key=lambda x: x[1]):
    label = '→'.join(c.label for c in loop)
    print(f"  Cluster {cid}: {label}")

Ends

---